In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import ltn
import pandas as pd
import numpy as np
import torch.nn as nn

In [4]:
base_model = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" 

"""
alternative fine tuned version:
model_id = "abhi9ab/DeepSeek-R1-Distill-Llama-8B-finance-v1"
"""

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Make my own fine-tuned Lora-Adapter
# model = PeftModel.from_pretrained(model, "./your-trained-lora-adapter")

model = model.eval()
print("model loaded!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

model loaded!


In [5]:
test_headlines = [
    "NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026."
]

In [6]:
for headline in test_headlines:
    prompt = f"<｜begin of sentence｜><｜User｜>Analyze the market sentiment of this news: {headline}<｜Assistant｜><｜thought｜>"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
<beginofsentence><｜User｜>Analyzethemarketsentimentofthisnews:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026.<｜Assistant｜><thought>Alright, so I'm trying to figure out how to analyze the market sentiment of this news: "NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 20-26." Hmm, okay, let's break this down step by step.

First, I know that NVIDIA is a big company in the tech industry, especially in graphics cards and AI. So, their earnings reports are tty important for the stock market and investor sentiment. The news says they beat Q4 expectations. That's a good sign because it means they're performing better than what people were expecting. Beating expectations can usually lead to positive sentiment because it shows strong performance and maybe even better-than-expected r

In [7]:
for headline in test_headlines:
    prompt = f"""
        You are a financial sentiment classifier.
        
        Analyze the headline and respond in EXACTLY this format:
        
        Sentiment: [Bullish, Bearish, or Neutral]
        Confidence: [number between 0 and 1]
        
        Rules:
        - Do NOT explain your reasoning
        - Output ONLY the two lines above
        - Be decisive
        
        Headline: {headline}
        """
            
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
Youareafinancialsentimentclassifier.AnalyzetheheadlineandrespondinEXACTLYthisformat:Sentiment:[Bullish,Bearish,orNeutral]Confidence:[numberbetween0and1]Rules:-DoNOTexplainyourreasoning-OutputONLYthetwolinesabove-BedecisiveHeadline:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026. NVIDIA'sQ4earningsexceedexpectations,buttheyalsohighlightedsupplychainconstraintsfor2026. So, the headline is about NVIDIA beating earnings expectations but also warning about supply chain constraints in 2026. Sentiment: Bullish Confidence: 0.8

Okay, so I need to analyze the headline "NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026." First, I'll break down the key components. NVIDIA is a major player in the tech industry, specifically in GPUs and AI, which are high-demand areas. Beating earnin

In [15]:
def generate_mock_market_data(num_samples=20):
    """
    Generates synthetic data:
    - Sentiment: [-1, 1]
    - Price_Trend: [-1, 1] (Relative to 200-day MA)
    - VIX: [10, 40] (Volatility)
    - is_material: boolean where True=Material (Real market move), False=Noisy
    """
    data = {
        'sentiment': np.random.uniform(-1, 1, num_samples),
        'price_trend': np.random.uniform(-1, 1, num_samples),
        'vix': np.random.uniform(10, 40, num_samples),
        'news_magnitude': np.random.uniform(0, 1, num_samples)
    }
    
    df = pd.DataFrame(data)
    
    # if magnitude is low but sentiment is high, it's not material (noise)
    df['is_material'] = (df['news_magnitude'] > 0.5) | (np.abs(df['sentiment']) > 0.7)
    
    return df

mock_df = generate_mock_market_data()

# convert to tensors for LTN
sentiment_t = torch.tensor(mock_df['sentiment'].values, dtype=torch.float32).unsqueeze(1)
vix_t = torch.tensor(mock_df['vix'].values, dtype=torch.float32).unsqueeze(1)
price_t = torch.tensor(mock_df['price_trend'].values, dtype=torch.float32).unsqueeze(1)

display(mock_df.head())

,sentiment,price_trend,vix,news_magnitude,is_material
0,0.001141,0.506872,17.837139,0.814446,True
1,-0.759658,-0.704768,14.243430,0.358411,True
2,-0.964120,0.754698,24.139250,0.523252,True
3,-0.872479,0.326222,36.237597,0.015459,True
4,-0.647003,-0.115782,39.373565,0.731575,True


In [17]:
# simple linear layer that reads a headline embedding and outputs sentiment (0 to 1)
sentiment_head = nn.Sequential(nn.Linear(4096, 1), nn.Sigmoid()).to("cuda").to(torch.bfloat16)
optimizer = torch.optim.AdamW(sentiment_head.parameters(), lr=0.01)

class MaterialSentimentPredicate(nn.Module):
    def forward(self, sentiment_output, magnitude):
        # symbolic logic: if magnitude < 0.5, force output to 0.5 (Neutral)
        is_material = (magnitude > 0.5).float()
        return sentiment_output * is_material + (0.5 * (1.0 - is_material))

is_bullish = ltn.Predicate(MaterialSentimentPredicate())

'''
- intialise with random numbers to test the maths
- news_vectors [-inf, +inf]
- magnitudes [0, 1]
- labels [0,1]
'''
num_samples = 20
news_vectors = torch.randn(num_samples, 4096).to("cuda").to(torch.bfloat16)
magnitudes = torch.rand(num_samples).unsqueeze(1).to("cuda").to(torch.bfloat16)
labels = (torch.rand(num_samples) > 0.5).float().unsqueeze(1).to("cuda").to(torch.bfloat16)

# train
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")

print("training neurosymbolic pipeline...")
for epoch in range(51):
    optimizer.zero_grad()
    
    # neural layer predicts sentiment
    raw_sentiment = sentiment_head(news_vectors)
    
    # wrap tensors into LTN Variables
    sentiment_var = ltn.Variable("sentiment", raw_sentiment)
    magnitude_var = ltn.Variable("magnitude", magnitudes)
    
    prediction = is_bullish(sentiment_var, magnitude_var)
    
    # MSE loss
    loss = torch.mean((prediction.value - labels)**2)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


# test: negative headline, is_material=False
test_vec = torch.randn(1, 4096).to("cuda").to(torch.bfloat16)
test_mag = torch.tensor([[0.1]]).to("cuda").to(torch.bfloat16)

# neural output
raw_sentiment_tensor = sentiment_head(test_vec)

# wrap in LTN variables
s_var = ltn.Variable("sentiment", raw_sentiment_tensor)
m_var = ltn.Variable("magnitude", test_mag)

# Pass Variables to Predicate (symbolic reasoning)
final_decision_obj = is_bullish(s_var, m_var)
final_decision = final_decision_obj.value.item()

print(f"Neural 'Gut Feeling' (Sentiment): {raw_sentiment_tensor.item():.4f}")
print(f"Symbolic Materiality Check: {trap_mag.item():.2f}")
print(f"Final Neurosymbolic Decision: {final_decision:.4f}")
print("the final decision is forced to 0.5 because the news is noise")

training neurosymbolic pipeline...
Epoch 0 | Loss: 0.2567
Epoch 10 | Loss: 0.1125
Epoch 20 | Loss: 0.1125
Epoch 30 | Loss: 0.1125
Epoch 40 | Loss: 0.1125
Epoch 50 | Loss: 0.1125
Neural 'Gut Feeling' (Sentiment): 0.0132
Symbolic Materiality Check: 0.10
Final Neurosymbolic Decision: 0.5000
the final decision is forced to 0.5 because the news is noise
